In [1]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

# This will create a default Spark session
spark = DataprocSparkSession.builder.getOrCreate()
# If you would like to customize the Spark session, please refer to the
# documentation at https://cloud.google.com/bigquery/docs/use-spark
#
# For example, if you need to use a different subnetwork, use the code below

# from google.cloud.dataproc_v1 import Session

# session = Session()
# session.environment_config.execution_config.subnetwork_uri = "<subnetwork_name>"
# spark = DataprocSparkSession.builder.dataprocSessionConfig(session).getOrCreate()

████████████████████████▌                                                       

In [2]:
from google.cloud import bigquery

# Inicializa o cliente do BigQuery padrão
client = bigquery.Client()

# Recupera 10% da base para uma exploração inicial
query = """
    SELECT *
    FROM `project-f1079166-4703-4cc3-975.raw_data.raw_accidents` TABLESAMPLE SYSTEM (10 PERCENT)
"""

# Traz para um DataFrame do Pandas para inspecionar de forma rápida
df_accidents = client.query(query).to_dataframe()
display(df_accidents.head())

,id_sinistro,tipo_registro,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,hora_sinistro,ano_mes_sinistro,dia_da_semana,turno,...,tp_sinistro_colisao_lateral,tp_sinistro_colisao_transversal,tp_sinistro_colisao_outros,tp_sinistro_choque,tp_sinistro_atrop_animal,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel
0,2484965,SINISTRO FATAL,2014-12-31,2014,12,31,17:30,2014/12,Quarta-feira,TARDE,...,None,None,None,None,None,None,None,None,None,None
1,2500135,SINISTRO FATAL,2015-01-04,2015,1,4,21:43,2015/01,Domingo,NOITE,...,None,None,None,None,None,None,None,None,None,None
2,2479836,SINISTRO FATAL,2015-01-11,2015,1,11,18:20,2015/01,Domingo,NOITE,...,None,None,None,None,None,None,None,None,None,None
3,2464854,SINISTRO FATAL,2015-01-14,2015,1,14,15:55,2015/01,Quarta-feira,TARDE,...,None,None,None,None,None,None,None,None,None,None
4,2490761,SINISTRO FATAL,2015-01-23,2015,1,23,05:00,2015/01,Sexta-feira,MADRUGADA,...,None,None,None,None,None,None,None,None,None,None


In [3]:
# Mostra o tipo de dado de cada coluna e se há valores nulos
print(df_accidents.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103845 entries, 0 to 103844
Data columns (total 50 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id_sinistro                         103845 non-null  Int64  
 1   tipo_registro                       103845 non-null  object 
 2   data_sinistro                       103845 non-null  dbdate 
 3   ano_sinistro                        103845 non-null  Int64  
 4   mes_sinistro                        103845 non-null  Int64  
 5   dia_sinistro                        103845 non-null  Int64  
 6   hora_sinistro                       103604 non-null  object 
 7   ano_mes_sinistro                    103845 non-null  object 
 8   dia_da_semana                       103845 non-null  object 
 9   turno                               103845 non-null  object 
 10  logradouro                          103585 non-null  object 
 11  numero_logradouro         

**Resumo:**



* O campo tp_sinistro_colisao_traseira apresenta todos os valores nulos, campo deverá ser removido nas próximas etapas. Também remover o campo tp_sinistro_nao_disponivel porque não retorna nenhum insight.
* Campos de coordernadas geograficas possuem somente 8% sem preenchimento, não impactando quando for plotar gráficos no Power BI.
* Colunas como qtd_pedestre, qtd_motocicleta, qtd_gravidade_fatal possuem muitos valores nulos aparentes. Ajustar campos nulos para 0, permitindo calculos mais precisos.



## Avaliação de Variáveis Categóricas

In [4]:
# Filtra apenas as colunas do tipo objeto (texto/categóricas)
colunas_categoricas = df_accidents.select_dtypes(include=['object']).columns

# Itera sobre cada coluna categórica mostrando a quantidade e os valores únicos
for coluna in colunas_categoricas:
    print(f"\n==================================================")
    print(f"COLUNA: {coluna}")
    print(f"Total de valores únicos: {df_accidents[coluna].nunique()}")
    print(f"Valores e frequências (Top 10):\n{df_accidents[coluna].value_counts().head(10)}")


COLUNA: tipo_registro
Total de valores únicos: 3
Valores e frequências (Top 10):
tipo_registro
SINISTRO NAO FATAL    64856
NOTIFICACAO           34536
SINISTRO FATAL         4453
Name: count, dtype: int64

COLUNA: hora_sinistro
Total de valores únicos: 1440
Valores e frequências (Top 10):
hora_sinistro
18:30    306
19:00    296
17:30    281
19:30    262
08:00    261
18:00    245
20:00    226
18:20    224
16:30    223
18:40    215
Name: count, dtype: int64

COLUNA: ano_mes_sinistro
Total de valores únicos: 142
Valores e frequências (Top 10):
ano_mes_sinistro
2023/06    1361
2023/07    1340
2023/05    1322
2023/08    1299
2023/12    1299
2023/10    1291
2023/09    1288
2024/03    1275
2024/08    1263
2022/10    1260
Name: count, dtype: int64

COLUNA: dia_da_semana
Total de valores únicos: 7
Valores e frequências (Top 10):
dia_da_semana
Sábado           16942
Sexta-feira      16596
Domingo          14707
Quinta-feira     14299
Quarta-feira     14082
Segunda-feira    13757
Terça-feira    

**Resumo:**


* Uniformizar colunas categóricas para evitar duplicações por diferenças de maiúsculas/minúsculas ou espaços extras.
* Substituir valores ausentes ou strings inválidas por null nos campos categoricos
* Não utilizar os campos de logradouro, conservacao, latitude, lodingute e municipio no modelo de Machine Learning devido a alta cardinalidade.


